In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

df = pd.read_csv("../data/features.csv")

print("Loaded features shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Loaded features shape: (12737, 14)

Missing values:
tweet_frequency     1000
account_age_days    1000
dtype: int64


   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 1.3 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/8.1 MB 1.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/8.1 MB 1.4 MB/s eta 0:00:06
   ------ --------------------------------- 1.3/8.1 MB 1.4 MB/s eta 0:00:05
   --------- ------------------------------ 1.8/8.1 MB 1.4 MB/s eta 0:00:05
   ---------- ----------------------------- 2.1/8.1 MB 1.4 MB/s eta 0:00:05
   ----------- ---------------------------- 2.4/8.1 MB 1.4 MB/s eta 0:00:05
   ------------ --------------------------- 2.6/8.1 MB 1.4 MB/s eta 0:00:05
   ------------ --------------------------- 2.6/8.1 MB 1.4 MB/s eta 0:00:05
   -------------- ------------------------- 2.9/8.1 MB 1.3 MB/s eta 0:00:05
   ---------------- -------------


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:

cols_to_fill = ["tweet_frequency", "account_age_days"]

for col in cols_to_fill:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} missing values with median: {median_val:.2f}")

print("\nMissing values after fix:")
print(df.isnull().sum().sum(), "total missing values remaining")

Filled tweet_frequency missing values with median: 0.06
Filled account_age_days missing values with median: 1433.00

Missing values after fix:
0 total missing values remaining


In [9]:

skewed_columns = [
    "statuses_count",
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "follower_friend_ratio",
    "tweet_frequency",
    "favourites_per_tweet",
    "listed_per_follower",
    "account_age_days"
]

for col in skewed_columns:
    df[col] = np.log1p(df[col])

print("Log scaling applied to:", skewed_columns)
print("\nStatuses count after log scaling:")
print(df["statuses_count"].describe().round(3))

Log scaling applied to: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'follower_friend_ratio', 'tweet_frequency', 'favourites_per_tweet', 'listed_per_follower', 'account_age_days']

Statuses count after log scaling:
count    12737.000
mean         5.222
std          2.727
min          0.000
25%          3.555
50%          4.174
75%          7.503
max         12.898
Name: statuses_count, dtype: float64


In [6]:
! pip install scikit-learn

^C


In [10]:


FEATURE_COLUMNS = [col for col in df.columns if col != "label"]

X = df[FEATURE_COLUMNS]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} accounts")
print(f"Test set:     {X_test.shape[0]} accounts")
print(f"\nTraining label breakdown:")
print(y_train.value_counts())
print(f"\nTest label breakdown:")
print(y_test.value_counts())

Training set: 10189 accounts
Test set:     2548 accounts

Training label breakdown:
label
1    7410
0    2779
Name: count, dtype: int64

Test label breakdown:
label
1    1853
0     695
Name: count, dtype: int64


In [11]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape:     {X_test_scaled.shape}")

Scaling complete.
Training set shape: (10189, 13)
Test set shape:     (2548, 13)


In [14]:
# Save the processed data so the training notebook can load it directly
# np.save stores numpy arrays (the scaled data)
np.save("../data/X_train.npy", X_train_scaled)
np.save("../data/X_test.npy", X_test_scaled)
np.save("../data/y_train.npy", y_train.values)
np.save("../data/y_test.npy", y_test.values)

# Save the scaler itself — we'll need it later when making predictions
# on new accounts (they need to be scaled the same way)
joblib.dump(scaler, "../models/scaler.joblib")

# Save the feature column names — useful later to know what order things are in
pd.Series(FEATURE_COLUMNS).to_csv("../data/feature_columns.csv", index=False)

print("Saved:")
print("  data/X_train.npy")
print("  data/X_test.npy")
print("  data/y_train.npy")
print("  data/y_test.npy")
print("  models/scaler.joblib")
print("  data/feature_columns.csv")

Saved:
  data/X_train.npy
  data/X_test.npy
  data/y_train.npy
  data/y_test.npy
  models/scaler.joblib
  data/feature_columns.csv


In [4]:
! pip install joblib


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
